# 📓 Notebook 1: Veri Toplama
## Zero-Day Odaklı Phishing Tespit Sistemi

Bu notebook'ta:
- PhishTank'tan phishing URL'leri toplayacağız
- Kaggle dataset'ini yükleyeceğiz
- İkisini birleştirip temizleyeceğiz
- Opsiyonel olarak WHOIS/SSL metadata çekeceğiz


In [1]:



import sys
import os
os.chdir('..')




sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_collector import run_data_collection, PhishTankCollector, KaggleLoader, DatasetMerger
from src.utils import summarize_dataset, print_section, save_dataframe, FIGURES_DIR

print('✅ Import başarılı')
print(f'Figures dizini: {FIGURES_DIR}')

✅ Import başarılı
Figures dizini: C:\Users\emira\OneDrive\Desktop\main projem\figures


## ⚙️ Konfigürasyon

In [2]:
CONFIG = {
    # Kaggle CSV yolu (eğer varsa)
    # Önerilen dataset: https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset
    
    'kaggle_csv': r'C:\Users\emira\OneDrive\Desktop\main projem\data\raw\kaggle_phishing.csv',
    
    # PhishTank API key (opsiyonel, ücretsiz kayıt: https://www.phishtank.com/api_info.php)
    'phishtank_api_key': None,
    
    # Toplam örnek sayısı
    'max_samples': 5000,


    
    # WHOIS/SSL metadata topla? (True ise çok yavaş olabilir)
    'collect_metadata': False,
    
    # Metadata toplanacaksa kaç URL için?
    'metadata_sample': 300,
    
    # Çıktı dosya adı
    'output_name': 'raw_dataset'
}

# Kaggle CSV var mı kontrol et
kaggle_path = Path(CONFIG['kaggle_csv'])
if kaggle_path.exists():
    print(f'✅ Kaggle CSV bulundu: {kaggle_path}')
else:
    print(f'ℹ️  Kaggle CSV bulunamadı ({kaggle_path})')
    print('   Demo veri ile devam edilecek')
    CONFIG['kaggle_csv'] = None

✅ Kaggle CSV bulundu: C:\Users\emira\OneDrive\Desktop\main projem\data\raw\kaggle_phishing.csv


## 📥 Veri Toplama

In [3]:
# Tüm pipeline'ı çalıştır
df = run_data_collection(
    kaggle_csv       = CONFIG['kaggle_csv'],
    phishtank_api_key = CONFIG['phishtank_api_key'],
    max_samples      = CONFIG['max_samples'],
    collect_metadata = CONFIG['collect_metadata'],
    metadata_sample  = CONFIG['metadata_sample'],
    output_name      = CONFIG['output_name']
)

print(f'\nDataset boyutu: {df.shape}')
df.head()

  VERİ TOPLAMA PIPELINE BAŞLIYOR
📥 PhishTank verisi çekiliyor...
✅ PhishTank: 2500 phishing URL alındı
📂 Kaggle dataset yükleniyor: C:\Users\emira\OneDrive\Desktop\main projem\data\raw\kaggle_phishing.csv
   Sütunlar: ['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_external_redirection', 'length_words_raw', 'char_repeat', 'shortest_words_raw', 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 'longest_word_host', 'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path',

,url,timestamp,label,source,domain_age_days,has_ssl,ssl_issuer,ssl_issuer_is_free,registrar,whois_available,registration_period_days,country
0,http://finance.yahoo.com/q?s=nok,2023-01-01 00:00:00+00:00,0,kaggle,-1,0,unknown,0,unknown,0,-1,unknown
1,https://techviral.net/malware-spreading-via-ppts/,2023-01-01 00:00:00+00:00,0,kaggle,-1,0,unknown,0,unknown,0,-1,unknown
2,https://www.bangbuzz.fr/tag/netflix/,2023-01-01 00:00:00+00:00,1,kaggle,-1,0,unknown,0,unknown,0,-1,unknown
3,https://www.youtube.com/watch?v=ldzcq3vukjs,2023-01-01 00:00:00+00:00,0,kaggle,-1,0,unknown,0,unknown,0,-1,unknown
4,https://herbpathy.com/uses-and-benefits-of-mim...,2023-01-01 00:00:00+00:00,0,kaggle,-1,0,unknown,0,unknown,0,-1,unknown


## 📊 Veri Keşfi (EDA)

In [4]:
# Dataset özeti
summarize_dataset(df, label_col='label')


════════════════════════════════════════════════════════════
  Dataset Özeti
════════════════════════════════════════════════════════════

Toplam örnek : 5,000
Feature sayısı: 11

Sınıf dağılımı:
  Legitimate (0): 2,500 (50.0%)
  Phishing (1): 2,500 (50.0%)

Eksik değerler:
Series([], dtype: int64)

Veri tipleri:
int64                  6
object                 5
datetime64[ns, UTC]    1
Name: count, dtype: int64


In [5]:
# Sınıf dağılımı
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pie chart
counts = df['label'].value_counts()
axes[0].pie(counts, labels=['Legitimate', 'Phishing'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Sınıf Dağılımı')

# Yıl bazlı dağılım
df['year'] = pd.to_datetime(df['timestamp']).dt.year
year_dist = df.groupby(['year', 'label']).size().unstack(fill_value=0)
year_dist.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'], alpha=0.8)
axes[1].set_title('Yıl Bazlı Dağılım')
axes[1].set_xlabel('Yıl')
axes[1].set_ylabel('Sayı')
axes[1].legend(['Legitimate', 'Phishing'])
axes[1].tick_params(axis='x', rotation=0)

# Kaynak dağılımı
source_counts = df['source'].value_counts()
axes[2].bar(source_counts.index, source_counts.values, color='#2196F3', alpha=0.8)
axes[2].set_title('Kaynak Dağılımı')
axes[2].set_xlabel('Kaynak')
axes[2].set_ylabel('Sayı')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Veri Seti Genel Görünümü', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik kaydedildi')

✅ Grafik kaydedildi


In [6]:
# URL uzunluk analizi
df['url_length'] = df['url'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(1, '#F44336', 'Phishing'), (0, '#4CAF50', 'Legitimate')]:
    subset = df[df['label'] == label]['url_length']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name)

axes[0].set_title('URL Uzunluk Dağılımı')
axes[0].set_xlabel('URL Uzunluğu')
axes[0].set_ylabel('Frekans')
axes[0].legend()
axes[0].set_xlim(0, 300)

# Box plot
data_to_plot = [df[df['label']==0]['url_length'], df[df['label']==1]['url_length']]
axes[1].boxplot(data_to_plot, labels=['Legitimate', 'Phishing'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_title('URL Uzunluk Box Plot')
axes[1].set_ylabel('URL Uzunluğu')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_url_length.png', dpi=150, bbox_inches='tight')
plt.show()

In [7]:
# Timestamp analizi - Zero-Day split'i görselleştir
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['month'] = df['timestamp'].dt.to_period('M')

monthly = df.groupby(['month', 'label']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(kind='area', ax=ax, alpha=0.6, color=['#4CAF50', '#F44336'])
ax.axvline(x=len(monthly[monthly.index < '2024-01']),
           color='blue', linestyle='--', linewidth=2, label='Train/Test Sınırı (2024)')
ax.set_title('Aylık URL Dağılımı (Zero-Day Split)', fontsize=13)
ax.set_xlabel('Tarih')
ax.set_ylabel('URL Sayısı')
ax.legend(['Legitimate', 'Phishing', 'Train/Test Sınırı'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_time_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Zero-day split görselleştirildi')

✅ Zero-day split görselleştirildi


In [8]:
# Kaydet
save_dataframe(df, 'raw_dataset')
print('\n✅ Notebook 1 tamamlandı!')
print('   Sonraki adım: 2_feature_engineering.ipynb')

[2026-02-21 15:57:26] INFO [utils] DataFrame kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\processed\raw_dataset (5000 satır)



✅ Notebook 1 tamamlandı!
   Sonraki adım: 2_feature_engineering.ipynb


In [9]:
import pandas as pd
test_df = pd.read_csv(r'C:\Users\emira\OneDrive\Desktop\main projem\data\raw\kaggle_phishing.csv')
print(test_df.shape)
print(test_df['status'].value_counts())

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64
